# 03 · Field Goal & PAT Success (IPW)

**Purpose:** Fit IPW-weighted cloglog model for FG/PAT success and produce diagnostics.

**Inputs:** Reference/fg_attempts_sample.csv, reports/attempt_p_hat_sample.csv

**Outputs:** reports/fg_success_calibration.csv, reports/fg_success_tail_plot.png

- [Parameters & Modes](#parameters--modes)
- [Imports](#imports--install-if-missing)
- [Utilities & Helpers](#utilities--helpers-≤40-lines-each)
- [Data Load & Peek](#data-load--peek)
- [Stage Logic](#stage-logic)
- [Artifacts](#artifacts)
- [Session Info](#session-info)


In [ ]:
# Parameters & Modes
SMOKE_MODE <- TRUE
FULL_MODE <- !SMOKE_MODE

reference_dir <- 'Reference'
data_dir <- 'data'
reports_dir <- 'reports'
config_path <- file.path('config', 'params.yaml')

if (!dir.exists(reports_dir)) {
  dir.create(reports_dir, recursive = TRUE)
}

params <- list(
  time_knots = c(60, 120, 300),
  p_clip_min = 0.05,
  p_clip_max = 0.95,
  tau_grid = c(0.03, 0.05, 0.07, 0.10),
  distance_cap = 65,
  yardline_spline_df = 5,
  weight_floor = 0.1,
  weight_cap = 10,
  late_game_threshold = 120,
  distance_spline_df = 6,
  wind_bins = c(0, 5, 10, 15, 25),
  default_p_hat = 0.5,
  default_m_hat = 0.65,
  overall_success_rate = 0.85
)

if (file.exists(config_path)) {
  tryCatch({
    config_params <- yaml::read_yaml(config_path)
    params <- utils::modifyList(params, config_params, keep.null = TRUE)
  }, error = function(e) message('Config read failed, using defaults: ', e$message))
}

list2env(params, envir = .GlobalEnv)
set.seed(101)


In [ ]:
# Imports — install if missing
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'mgcv', 'splines', 'glmmTMB', 'pROC', 'yaml', 'scales'
)

install_if_missing <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = 'https://cloud.r-project.org')
  }
}

invisible(purrr::walk(dependencies, install_if_missing))

library(dplyr)
library(tibble)
library(tidyr)
library(readr)
library(stringr)
library(purrr)
library(ggplot2)
library(mgcv)
library(splines)
library(glmmTMB)
library(pROC)
library(yaml)
library(scales)


### Function Index
- `get_schema()` — quick schema glance at data frames.
- `add_time_features()` — derive score/time convenience features.
- `ensure_columns()` — add fallback columns with default values.
- `clip_weights()` — enforce weight floor/cap.
- `stabilize_weights()` — compute stabilized attempt weights.
- `calc_brier()` — calculate (weighted) Brier score.
- `plot_calibration()` — convenience calibration scatter/smoother.


In [ ]:
# Utilities & Helpers (≤40 lines each)
get_schema <- function(df, n = 5) {
  tibble::tibble(
    name = names(df),
    class = purrr::map_chr(df, ~ paste(class(.x), collapse = '/')),
    example = purrr::map_chr(df, ~ paste(head(.x, n), collapse = ', '))
  )
}

add_time_features <- function(df) {
  df %>%
    mutate(
      score_diff = dplyr::coalesce(score_diff, score_differential, 0),
      time_remaining = dplyr::coalesce(game_seconds_remaining, quarter_seconds_remaining, 0),
      log_time_remaining = log1p(time_remaining),
      late_game = time_remaining <= late_game_threshold,
      one_score = abs(score_diff) <= 8
    )
}

ensure_columns <- function(df, defaults) {
  for (nm in names(defaults)) {
    if (!nm %in% names(df)) {
      df[[nm]] <- defaults[[nm]]
    }
  }
  df
}

clip_weights <- function(w, floor = weight_floor, cap = weight_cap) {
  pmin(pmax(w, floor), cap)
}

stabilize_weights <- function(p_hat, base_rate) {
  clip_weights(base_rate / p_hat)
}

calc_brier <- function(actual, predicted, weights = NULL) {
  if (is.null(weights)) {
    mean((predicted - actual) ^ 2)
  } else {
    sum(weights * (predicted - actual) ^ 2) / sum(weights)
  }
}

plot_calibration <- function(df, prob_col, outcome_col, group_col, path) {
  plot <- ggplot(df, aes_string(x = prob_col, y = outcome_col, color = group_col)) +
    geom_point(alpha = 0.4) +
    geom_smooth(method = 'loess', se = FALSE) +
    labs(title = 'Calibration', x = 'Predicted', y = 'Observed')
  ggsave(path, plot = plot, width = 6, height = 4, dpi = 150)
  invisible(plot)
}


In [ ]:
# Data Load & Peek
fg_attempts <- readr::read_csv(file.path(reference_dir, 'fg_attempts_sample.csv'), show_col_types = FALSE)
fg_attempts <- ensure_columns(fg_attempts, list(
  season = 2015L,
  play_id = '0',
  game_id = '0',
  kick_distance = 35,
  time_remaining = 120,
  wind = 5,
  kick_made = 1,
  is_pat = FALSE
))

attempt_preds_path <- file.path(reports_dir, 'attempt_p_hat_sample.csv')
if (file.exists(attempt_preds_path)) {
  attempt_preds <- readr::read_csv(attempt_preds_path, show_col_types = FALSE)
  fg_attempts <- fg_attempts %>% left_join(attempt_preds, by = c('game_id', 'play_id', 'season'))
}

if (SMOKE_MODE) {
  fg_attempts <- fg_attempts %>% slice_sample(n = min(4000, n()))
}

fg_attempts <- fg_attempts %>% add_time_features()
get_schema(fg_attempts) %>% print(n = 10)


In [ ]:
# Stage Logic — Success Model
## TODO: replace mock predictions with glmmTMB fit and diagnostics.

fg <- fg_attempts %>%
  mutate(
    is_pat = is_pat %in% c(1, TRUE, '1', 'TRUE'),
    success = ifelse(!is.na(kick_made), kick_made, ifelse(field_goal_result == 'made', 1, 0)),
    w = ifelse(is_pat, 1, ifelse(!is.na(w), w, 1)),
    distance_capped = pmin(kick_distance, distance_cap),
    late_game = time_remaining <= late_game_threshold,
    wind_bucket = cut(wind, breaks = c(-Inf, wind_bins, Inf), include.lowest = TRUE)
  )

set.seed(123)
fg_results <- fg %>%
  mutate(
    m_hat = plogis(rnorm(n(), qlogis(overall_success_rate), 0.5)),
    m_hat = pmin(pmax(m_hat, 1e-3), 1 - 1e-3)
  )

calibration <- fg_results %>%
  mutate(distance_bin = cut(distance_capped, breaks = seq(10, 65, by = 5), include.lowest = TRUE)) %>%
  group_by(distance_bin) %>%
  summarise(
    n = dplyr::n(),
    rate = mean(success, na.rm = TRUE),
    mean_pred = mean(m_hat),
    .groups = 'drop'
  )

readr::write_csv(calibration, file.path(reports_dir, 'fg_success_calibration.csv'))

tail_plot <- ggplot(fg_results, aes(x = distance_capped, y = m_hat)) +
  geom_line(color = '#253494') +
  labs(title = 'Tail Behavior (50–60 yds)', x = 'Distance (yds)', y = 'Predicted make %') +
  coord_cartesian(xlim = c(45, 65))

ggplot2::ggsave(
  filename = file.path(reports_dir, 'fg_success_tail_plot.png'),
  plot = tail_plot,
  width = 7,
  height = 4,
  dpi = 150
)

# Artifact note
message('Produced placeholder success calibration table and tail plot.')


### Artifacts
- See generated files under `reports/` when the notebook is executed.


In [ ]:
# Session Info
info <- capture.output(sessionInfo())
readr::write_lines(info, file.path(reports_dir, 'session_info.txt'), append = TRUE)
cat(info, sep = '
')
